# Análisis exploratorio de los datasets

## 1. Descargamos los datos

In [1]:
import os
import pandas as pd

users_path = "https://raw.githubusercontent.com/gefero/ecyt_lcd_intro_nlp/refs/heads/main/TFI/perfiles_usuarios.csv"
users_df = None

if not os.path.exists("data/perfiles_usuarios.csv"):
    users_df = pd.read_csv(users_path)
    users_df.to_csv("data/perfiles_usuarios.csv", index=False)
else:
    users_df = pd.read_csv("data/perfiles_usuarios.csv")

In [2]:
plots_path = "hf://datasets/mathigatti/spanish_imdb_synopsis/plots.csv"
df = None

if not os.path.exists("data/plots.csv"):
    df = pd.read_csv(plots_path)
    df.to_csv("data/plots.csv", index=False)
else:
    df = pd.read_csv("data/plots.csv")

## 2. Descripciones básicas

In [3]:
users_df.head()

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...


In [4]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           14 non-null     str  
 1   nombre       14 non-null     str  
 2   tipo_perfil  14 non-null     str  
 3   pelicula_1   14 non-null     str  
 4   pelicula_2   14 non-null     str  
 5   pelicula_3   14 non-null     str  
 6   pelicula_4   14 non-null     str  
 7   pelicula_5   14 non-null     str  
 8   query        14 non-null     str  
dtypes: str(9)
memory usage: 1.1 KB


In [5]:
df.head()

,description,keywords,genre,year,name,director
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4967 entries, 0 to 4966
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   description  4967 non-null   str    
 1   keywords     4967 non-null   str    
 2   genre        4967 non-null   str    
 3   year         4933 non-null   float64
 4   name         4967 non-null   str    
 5   director     4467 non-null   str    
dtypes: float64(1), str(5)
memory usage: 233.0 KB


# 3. Vemos si las películas de los usuarios están en el dataset de HF

In [7]:
# Hacer una lista plana con las películas de cada usuario, sin duplicados
# Las películas están en las columnas "pelicula_1", "pelicula_2", "pelicula_3", etc.
# Hay una película por celda
peliculas = set()
for col in users_df.columns:
    if col.startswith("pelicula_"):
        peliculas.update(users_df[col].dropna().unique())
peliculas = list(peliculas)
print(f"Total de películas únicas: {len(peliculas)}")

Total de películas únicas: 64


In [9]:
# Quiero ver que las películas únicas de los usuarios estén en el dataset de plots
peliculas_en_plots = set(df["name"].unique())
peliculas_no_en_plots = [p for p in peliculas if p not in peliculas_en_plots]
print(f"Películas de usuarios que no están en plots: {peliculas_no_en_plots}")


Películas de usuarios que no están en plots: ['Walker Texas Ranger', 'Amélie', 'Una mente brillante', 'Paddington', 'Kill Bill', 'Rec', 'El exorcista', 'Intocable', 'Mamma Mia!', 'El secreto de sus ojos']


In [11]:
# A partir de las películas que no estan en plots, qué usuarios las tienen en su perfil?
usuarios_con_peliculas_no_en_plots = set()
for col in users_df.columns:
    if col.startswith("pelicula_"):
        usuarios_con_peliculas_no_en_plots.update(users_df[users_df[col].isin(peliculas_no_en_plots)]["nombre"])
print(f"Usuarios con películas no en plots: {usuarios_con_peliculas_no_en_plots}")

Usuarios con películas no en plots: {'Mariana', 'Nicolás', 'Paula', 'Diego', 'Julián'}


### Estrategia clase 1

Por un lado, vamos a usar la descripción de las películas y entrenar un modelo Word2Vec como en la clase 1 de la Unidad 4 (actividad de los Simpsons). Por otro lado, usando TF-IDF, tomamos los términos más relevantes de las queries de los usuarios para usar como input en la busqueda de relaciones positivas del modelo entrenado Word2Vec

### Estrategia clase 2

Para la segunda parte o segunda representación vamos a usar una estrategia de LLMs. La idea es que el LLM entrenado con el corpus de plots reciba la query del usuario y recomiende la siguiente película.